<h2>Food Recipe Website Scrape 1</h2>

<h3>Contents</h3>
<ul>
    <li><a id='#intro'>Introduction</a></li>
    <li><a id='#testing'>Testing</a></li>
    <li><a id='#code'>Scrapcode</a></li>
</ul>


<h3>Introduction</h3>
<a id='intro'></a>

<p>To be added</p>

In [26]:
#importing necessary packages

from bs4 import BeautifulSoup
import lxml
import requests
import numpy as np
import pandas as pd
print("packages imported successfully")

packages imported successfully


In [27]:
#assigning site to a variable
website = "https://www.bbcgoodfood.com"
websiteSearchPattern = "https://www.bbcgoodfood.com/search?q="

### Testing Section
<a id='testing'></a>
<p>This section primarily looks at testing what can be scraped from the website. Different aspects of the site will be scraped to see how they are done before proceeding to write the whole code to scrape multiple items from the site.</p>

In [28]:
#use requests package to get html content of the site and parse it in beautifulsoup with lxml
site_request = requests.get(websiteSearchPattern+"chicken")
site_soup = BeautifulSoup(site_request.text,'lxml')

#writing contents to a file
with open('Site content.txt','w') as file:
    file.write(site_soup.prettify())
    file.close()

KeyboardInterrupt: 

In [ ]:
#searching content for search results
recipes = site_soup.find_all('article')

Getting information from the first result

In [ ]:
firstRecipe = recipes[0]

#getting name of recipe
recipeName = firstRecipe.find('h2',class_='heading-4').text

#getting the ratings
recipeRating = firstRecipe.find(class_='sr-only').text.split()[4]
reciptRatingCount = firstRecipe.find(class_='rating__count-text body-copy-small').text.split()[0]

print(f"The name of the recipe is {recipeName} and it has been rated up to {reciptRatingCount} times to obtain a rating of {recipeRating} out of 5")

The name of the recipe is Chicken & chorizo jambalaya and it has been rated up to 2668 times to obtain a rating of 4.8 out of 5


In [ ]:
#get recipe link

recipeLink = firstRecipe.find(class_='card__section card__media').a['href']
print("Link for recipe is " + recipeLink)

recipeSite = requests.get(website+recipeLink)
recipeSoup = BeautifulSoup(recipeSite.text,'lxml')

Link for recipe is /recipes/chicken-chorizo-jambalaya


In [ ]:
#information from recipe link

recipePrepTime = int(recipeSoup.find_all('time')[0].text.split()[0])
recipeCookTime = int(recipeSoup.find_all('time')[1].text.split()[0])

print("Prep time: "+ str(recipePrepTime))
print("Cooking time: " + str(recipeCookTime))
print("Total time: " + str(recipePrepTime+recipeCookTime))

Prep time: 10
Cooking time: 45
Total time: 55


In [ ]:
#get and store recipe image
recipeImg = recipeSoup.find(class_='post recipe').find('img', class_='image__img')['src']
ImgContent = requests.get(recipeImg).content

with open(recipeName+".png",'wb') as file:
    file.write(ImgContent)
    file.close()

In [ ]:
recipeSoup.find_all('td')

[<td class="key-value-blocks__prefix"></td>,
 <td class="key-value-blocks__key">kcal</td>,
 <td class="key-value-blocks__value">445</td>,
 <td class="key-value-blocks__prefix"></td>,
 <td class="key-value-blocks__key">fat</td>,
 <td class="key-value-blocks__value">10<!-- -->g</td>,
 <td class="key-value-blocks__prefix"></td>,
 <td class="key-value-blocks__key">saturates</td>,
 <td class="key-value-blocks__value">3<!-- -->g</td>,
 <td class="key-value-blocks__prefix"></td>,
 <td class="key-value-blocks__key">carbs</td>,
 <td class="key-value-blocks__value">64<!-- -->g</td>,
 <td class="key-value-blocks__prefix"></td>,
 <td class="key-value-blocks__key">sugars</td>,
 <td class="key-value-blocks__value">7<!-- -->g</td>,
 <td class="key-value-blocks__prefix"></td>,
 <td class="key-value-blocks__key">fibre</td>,
 <td class="key-value-blocks__value">2<!-- -->g</td>,
 <td class="key-value-blocks__prefix"></td>,
 <td class="key-value-blocks__key">protein</td>,
 <td class="key-value-blocks__val

In [ ]:
test = recipeSoup.find_all('td')
print("number of items: "+str(len(test)))
for row in test:
    if row.text == 'kcal':
        print(row.find_next('td').text)



number of items: 24
445


Different Search patterns

<li>By Rating: https://www.bbcgoodfood.com/search?q=chicken&tab=recipe&sort=rating</li>
<li>By Relevance: https://www.bbcgoodfood.com/search?q=chicken&tab=recipe&sort=relevant</li>
<li>By Fastest Cook time: https://www.bbcgoodfood.com/search?q=chicken&tab=recipe&sort=quickest</li>
<li>By Latest added: https://www.bbcgoodfood.com/search?q=chicken&tab=recipe&sort=published</li>

### ScarpCode <a id='code'></a>

<p>The code that will be written will be making use of the search pattern by rating and will only scrape the first page as it contains 30 results</p>

In [37]:
'''
For this code, the search pattern used will be By Rating and only the first page will be scraped which should be the most rated foods
on the platform. The code will start by requesting for a food to search for and the results will be stored in a dictionary which will later on
be converted to a datatable using pandas.
'''
website = "https://www.bbcgoodfood.com"
search = input("Enter a meal/dish you want to search for: ")
websiteSearchPattern = f'https://www.bbcgoodfood.com/search?q={search}&tab=recipe&sort=rating'

#defining the dictionary
ScrapeResults = {}

site_request = requests.get(websiteSearchPattern)
site_soup = BeautifulSoup(site_request.text,'lxml')


recipes_results = site_soup.find_all('article')

#print("The result type is "+ str(type(recipes_results)) +"\n The number of results is " + str(len(recipes_results)))

count = 1
try:
    #looping through all the results
    for index,recipe in enumerate(recipes_results):


        innerResult = {}

        #print("Looking at recipe " + str(index))
        print("Looking at recipe " + str(count))
        innerResult["Name"] = recipe.find('h2',class_='heading-4').text
        innerResult["Rating"] = recipe.find(class_='sr-only').text.split()[4]
        innerResult["Review Count"] = recipe.find(class_='rating__count-text body-copy-small').text.split()[0]

        innerResult["link"] = website+recipe.find(class_='card__section card__media').a['href']
        print("Link for recipe is " + innerResult["link"])

        #dive into each recipe
        print('getting more information for recipe unique link')
        recipeSite = requests.get(innerResult["link"])
        recipeSoup = BeautifulSoup(recipeSite.text,'lxml')
        
        for timeSearch in recipeSoup.find_all(class__='body-copy-bold mr-xxs'):
            if "Prep" in timeSearch.text:
                print("Continue with search")
                innerResult["Prep time"] = int(recipeSoup.find_all('time')[0].text.split()[0])
                innerResult["Cooking time"] = int(recipeSoup.find_all('time')[1].text.split()[0])
                innerResult["Total time"] = innerResult["Prep time"] + innerResult["Cooking time"]
            else:
                print("different format encountered")
                innerResult["Total time"] = int(recipeSoup.find_all('time')[0].text.split()[0])
        


        tablesearch = recipeSoup.find_all('td')
        print("number of items: "+str(len(tablesearch)))
        for row in tablesearch:
            if row.text == 'kcal':
                calories = row.find_next('td').text
                print("number of calories is " + calories + "kcal")
                innerResult["kcal"] = calories

        ScrapeResults[index] = innerResult
        count += 1

except:
        print("Exception occurred as item is not a recipe card")

Looking at recipe 1
Link for recipe is https://www.bbcgoodfood.com/recipes/chicken-chorizo-jambalaya
getting more information for recipe unique link
number of items: 24
number of calories is 445kcal
Looking at recipe 2
Link for recipe is https://www.bbcgoodfood.com/recipes/red-lentil-chickpea-chilli-soup
getting more information for recipe unique link
number of items: 24
number of calories is 222kcal
Looking at recipe 3
Link for recipe is https://www.bbcgoodfood.com/recipes/chicken-noodle-soup
getting more information for recipe unique link
number of items: 24
number of calories is 217kcal
Looking at recipe 4
Link for recipe is https://www.bbcgoodfood.com/recipes/home-style-chicken-curry
getting more information for recipe unique link
number of items: 24
number of calories is 382kcal
Looking at recipe 5
Link for recipe is https://www.bbcgoodfood.com/recipes/chicken-biryani
getting more information for recipe unique link
number of items: 16
number of calories is 617kcal
Looking at recip

In [36]:
recipe

<article class="card text-align-left card--horizontal card--transparent" data-component="Card" data-index="1" data-item-name="5 issues for just £5" data-variant="horizontal"><div class="card__section card__media"><div class="d-block p-relative"><div class="p-absolute card__media-buttons pa-xs"></div></div><a aria-label="View 5 issues for just £5" class="link d-block p-relative card__image-container" data-component="Link" href="https://www.buysubscriptions.com/print/good-food-magazine-subscription?promo=GFBS824&amp;utm_medium=brandsite&amp;utm_source=goodfood.com&amp;utm_campaign=cook_the_cover_gfbs824&amp;utm_content=footer-widget&amp;style=brand"><div class="image chromatic-ignore card__img image--fluid image--scaled-up"><div class="image__container"><picture class="image__picture" height="84.444" width="93"><source sizes="93px" srcset="https://images.immediate.co.uk/production/volatile/sites/30/2021/11/632001115-gdfd-brandsite-180x180px-728e854.jpg?quality=90&amp;webp=true&amp;resize

In [32]:
ScrapeResults

{0: {'Name': 'Chicken & chorizo jambalaya',
  'Rating': '4.8',
  'Review Count': '2668',
  'link': 'https://www.bbcgoodfood.com/recipes/chicken-chorizo-jambalaya',
  'kcal': '445'},
 1: {'Name': 'Red lentil, chickpea & chilli soup',
  'Rating': '4.8',
  'Review Count': '850',
  'link': 'https://www.bbcgoodfood.com/recipes/red-lentil-chickpea-chilli-soup',
  'kcal': '222'},
 2: {'Name': 'Chicken noodle soup',
  'Rating': '4.8',
  'Review Count': '792',
  'link': 'https://www.bbcgoodfood.com/recipes/chicken-noodle-soup',
  'kcal': '217'},
 3: {'Name': 'Home-style chicken curry',
  'Rating': '4.7',
  'Review Count': '774',
  'link': 'https://www.bbcgoodfood.com/recipes/home-style-chicken-curry',
  'kcal': '382'},
 4: {'Name': 'Chicken biryani',
  'Rating': '4.7',
  'Review Count': '754',
  'link': 'https://www.bbcgoodfood.com/recipes/chicken-biryani',
  'kcal': '617'},
 5: {'Name': 'Summer-in-winter chicken',
  'Rating': '4.5',
  'Review Count': '820',
  'link': 'https://www.bbcgoodfood.c

This next section will be making use of pandas library to convert the dictionary into a table and see if any analytics can be done

In [35]:
pd.DataFrame.from_dict(ScrapeResults,orient='index')

,Name,Rating,Review Count,link,kcal
0,Chicken & chorizo jambalaya,4.8,2668,https://www.bbcgoodfood.com/recipes/chicken-ch...,445
1,"Red lentil, chickpea & chilli soup",4.8,850,https://www.bbcgoodfood.com/recipes/red-lentil...,222
2,Chicken noodle soup,4.8,792,https://www.bbcgoodfood.com/recipes/chicken-no...,217
3,Home-style chicken curry,4.7,774,https://www.bbcgoodfood.com/recipes/home-style...,382
4,Chicken biryani,4.7,754,https://www.bbcgoodfood.com/recipes/chicken-bi...,617
5,Summer-in-winter chicken,4.5,820,https://www.bbcgoodfood.com/recipes/summer-win...,262
6,Chicken pasta bake,4.7,724,https://www.bbcgoodfood.com/recipes/chicken-pa...,575
7,Carrot & coriander soup,4.7,677,https://www.bbcgoodfood.com/recipes/carrot-cor...,115
8,One-pot chicken chasseur,4.8,576,https://www.bbcgoodfood.com/recipes/one-pot-ch...,439
9,Mustard-stuffed chicken,4.6,646,https://www.bbcgoodfood.com/recipes/mustard-st...,367
